In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from datetime import datetime
import sqlite3

import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [2]:
# %%
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

BASE_URL = "https://www.edpb.europa.eu/news/news_en"
print("✅ Scraper do EDPB News pronto!")

✅ Scraper do EDPB News pronto!


In [3]:
DATABASE_NAME = "internet_governance_news.db"

def create_database():
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS articles (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            title TEXT,
            date TEXT,
            author TEXT,
            url TEXT UNIQUE,
            source TEXT
        )
    """)
    conn.commit()
    conn.close()
    print("✅ Banco e tabela 'articles' prontos!")

create_database()


✅ Banco e tabela 'articles' prontos!


In [4]:
def insert_article(title, date, author, url, source):
    conn = sqlite3.connect(DATABASE_NAME)
    cursor = conn.cursor()
    try:
        cursor.execute("""
            INSERT INTO articles (title, date, author, url, source)
            VALUES (?, ?, ?, ?, ?)
        """, (title, date, author, url, source))
        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

In [5]:
# %%
def load_articles_from_db():
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("""
        SELECT *
        FROM articles
        ORDER BY date DESC
    """, conn)
    conn.close()
    return df

df_db = load_articles_from_db()
display(df_db.head())
print(f"📦 Total no banco: {len(df_db)} registros")


,id,title,date,author,url,source
0,792,Nueva sentencia de la Corte Europea determina ...,"9 septiembre, 2021",Observacom,https://www.observacom.org/nueva-sentencia-de-...,Observacom
1,1100,Gigantes tecnológicos centralizan el poder de ...,"9 septiembre, 2020",Observacom,https://www.observacom.org/gigantes-tecnologic...,Observacom
2,1356,Lobby de grupos evangélicos para reformar ley ...,"9 septiembre, 2019",Observacom,https://www.observacom.org/lobby-de-grupos-eva...,Observacom
3,1960,AMARC señala falta de voluntad política para r...,"9 septiembre, 2016",Observacom,https://www.observacom.org/amarc-senala-falta-...,Observacom
4,236,"Meta, Google y TikTok concentran el 70% del tr...","9 octubre, 2024",Observacom,https://www.observacom.org/meta-google-y-tikto...,Observacom


📦 Total no banco: 3089 registros


In [6]:
# %%
def montar_url(pagina):
    """Monta a URL paginada do EDPB News (page=0 é a primeira)."""
    if pagina == 0:
        return f"{BASE_URL}?news_type=1"
    return f"{BASE_URL}?news_type=1&page={pagina}"

def extrair_paragrafos(url):
    """Acessa a página da notícia e extrai os parágrafos principais."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
        ps = soup.find_all("p")
        textos = [
            p.get_text(strip=True)
            for p in ps
            if len(p.get_text(strip=True).split()) > 10
        ]
        return textos[:5] if textos else ["NA"]
    except:
        return ["NA"]

In [7]:
# %%
noticias = []
TOTAL_PAGES = 30  # páginas 0..29 (30 páginas, ~10 itens cada)

for pagina in range(0, TOTAL_PAGES):
    url = montar_url(pagina)
    print(f"📄 Coletando página {pagina + 1}/{TOTAL_PAGES}: {url}")

    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code != 200:
            print(f"⚠️ Erro HTTP {r.status_code} — parando.")
            break
    except Exception as e:
        print(f"⚠️ Erro de conexão: {e}")
        break

    soup = BeautifulSoup(r.text, "html.parser")

    # ===== Seletores para EDPB (Drupal)
    itens = soup.select(".views-row")
    if not itens:
        print("🏁 Sem mais resultados — fim da coleta.")
        break

    print(f"   {len(itens)} notícias encontradas")

    for item in itens:
        # título & link
        titulo_tag = item.select_one("h4 a")
        if not titulo_tag:
            titulo_tag = item.select_one("h3 a")
        if not titulo_tag:
            continue

        titulo = titulo_tag.get_text(strip=True)
        link = titulo_tag.get("href", "")
        if link.startswith("/"):
            link = f"https://www.edpb.europa.eu{link}"

        # data
        date_tag = item.select_one("time")
        if date_tag:
            data_raw = date_tag.get("datetime", date_tag.get_text(strip=True))
        else:
            # fallback: procura span com classe date
            date_span = item.select_one(".date-display-single, .datetime")
            data_raw = date_span.get_text(strip=True) if date_span else "NA"

        # extrai parágrafos da notícia
        paragrafos = extrair_paragrafos(link)

        # acumula
        noticias.append({
            "titulo": titulo,
            "data": data_raw,
            "link": link,
            "paragrafos": " || ".join(paragrafos),
            "fonte": "EDPB"
        })

        # grava no banco
        insert_article(
            title=titulo,
            date=data_raw,
            author="EDPB",
            url=link,
            source="EDPB"
        )

    time.sleep(1)

print(f"\n✅ Total coletado: {len(noticias)} notícias")

df_edpb = pd.DataFrame(noticias)
display(df_edpb.head())

📄 Coletando página 1/30: https://www.edpb.europa.eu/news/news_en?news_type=1
   10 notícias encontradas
📄 Coletando página 2/30: https://www.edpb.europa.eu/news/news_en?news_type=1&page=1
   10 notícias encontradas
📄 Coletando página 3/30: https://www.edpb.europa.eu/news/news_en?news_type=1&page=2
   10 notícias encontradas
📄 Coletando página 4/30: https://www.edpb.europa.eu/news/news_en?news_type=1&page=3
   10 notícias encontradas
📄 Coletando página 5/30: https://www.edpb.europa.eu/news/news_en?news_type=1&page=4
   10 notícias encontradas
📄 Coletando página 6/30: https://www.edpb.europa.eu/news/news_en?news_type=1&page=5
   10 notícias encontradas
📄 Coletando página 7/30: https://www.edpb.europa.eu/news/news_en?news_type=1&page=6
   10 notícias encontradas
📄 Coletando página 8/30: https://www.edpb.europa.eu/news/news_en?news_type=1&page=7
   10 notícias encontradas
📄 Coletando página 9/30: https://www.edpb.europa.eu/news/news_en?news_type=1&page=8
   10 notícias encontradas
📄 Coleta

,titulo,data,link,paragrafos,fonte
0,EDPB brings clarity to data processing for sci...,NA,https://www.edpb.europa.eu/news/news/2026/edpb...,"Brussels, 16 April – During its latest plenary...",EDPB
1,Enhancing compliance and consistency: EDPB ado...,NA,https://www.edpb.europa.eu/news/news/2026/enha...,"Brussels, 14 April - In line with theEDPB’s He...",EDPB
2,EDPB annual report 2025: supporting stakeholde...,NA,https://www.edpb.europa.eu/news/news/2026/edpb...,"Brussels, 09 April - The European Data Protect...",EDPB
3,EDPB conference on cross-regulatory cooperatio...,NA,https://www.edpb.europa.eu/news/news/2026/edpb...,"Brussels, 23 March - On 17 March 2026, theEDPB...",EDPB
4,EDPB and EDPS support strengthening EU’s cyber...,NA,https://www.edpb.europa.eu/news/news/2026/edpb...,"Brussels, 19 March 2026 – The European Data Pr...",EDPB


In [8]:
def load_articles():
    conn = sqlite3.connect(DATABASE_NAME)
    df = pd.read_sql("""
        SELECT * FROM articles
        ORDER BY date DESC
    """, conn)
    conn.close()
    return df

df_db = load_articles()
print(f"📦 Total no banco: {len(df_db)} registros")
display(df_db.head(20))

📦 Total no banco: 3387 registros


,id,title,date,author,url,source
0,3090,EDPB brings clarity to data processing for sci...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB
1,3091,Enhancing compliance and consistency: EDPB ado...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/enha...,EDPB
2,3092,EDPB annual report 2025: supporting stakeholde...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB
3,3093,EDPB conference on cross-regulatory cooperatio...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB
4,3094,EDPB and EDPS support strengthening EU’s cyber...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB
5,3095,CEF 2026: EDPB launches coordinated enforcemen...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/cef-...,EDPB
6,3096,EDPB and EDPS support harmonisation of clinica...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/edpb...,EDPB
7,3097,Stakeholder event on political advertising: ag...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/stak...,EDPB
8,3098,Conference on cross-regulatory cooperation in ...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/conf...,EDPB
9,3099,AI-generated imagery and protection of privacy...,NA,EDPB,https://www.edpb.europa.eu/news/news/2026/ai-g...,EDPB


In [9]:
keywords = ['digital', 'internet', 'IA', 'tecnologia', 'dados', 'privacidade',
            'data', 'privacy', 'GDPR', 'AI', 'protection', 'enforcement']
pattern = r'|'.join(keywords)

df_filt = df_edpb[
    df_edpb['titulo'].str.contains(pattern, case=False, na=False, regex=True) |
    df_edpb['paragrafos'].str.contains(pattern, case=False, na=False, regex=True)
].copy()

print(f"{len(df_filt)} notícias filtradas (de {len(df_edpb)})")
display(df_filt.head())

272 notícias filtradas (de 298)


,titulo,data,link,paragrafos,fonte
0,EDPB brings clarity to data processing for sci...,NA,https://www.edpb.europa.eu/news/news/2026/edpb...,"Brussels, 16 April – During its latest plenary...",EDPB
1,Enhancing compliance and consistency: EDPB ado...,NA,https://www.edpb.europa.eu/news/news/2026/enha...,"Brussels, 14 April - In line with theEDPB’s He...",EDPB
2,EDPB annual report 2025: supporting stakeholde...,NA,https://www.edpb.europa.eu/news/news/2026/edpb...,"Brussels, 09 April - The European Data Protect...",EDPB
3,EDPB conference on cross-regulatory cooperatio...,NA,https://www.edpb.europa.eu/news/news/2026/edpb...,"Brussels, 23 March - On 17 March 2026, theEDPB...",EDPB
4,EDPB and EDPS support strengthening EU’s cyber...,NA,https://www.edpb.europa.eu/news/news/2026/edpb...,"Brussels, 19 March 2026 – The European Data Pr...",EDPB


In [10]:
def plot_charts(df):
    if df.empty:
        print("❌ Sem dados para gráficos")
        return

    # ------------------------------
    # Top 15
    # ------------------------------
    top15 = df.head(15).copy()
    top15['rank'] = range(1, len(top15) + 1)

    fig1 = px.bar(
        top15,
        x='rank',
        y='title',
        orientation='h',
        title='Top 15 Notícias – Internet Governance'
    )
    fig1.update_layout(height=600)
    fig1.show()

    # ------------------------------
    # Pizza por Fonte (BANCO)
    # ------------------------------
    # Fonte
    source_count = df["source"].value_counts().reset_index()
    source_count.columns = ["source", "count"]

    fig2 = px.pie(
        source_count,
        names="source",
        values="count",
        title="Distribuição por Fonte"
    )
    fig2.show()

    # ------------------------------
    # Nuvem de Palavras
    # ------------------------------
    text = ' '.join(df['title'].astype(str)).lower()
    words = re.findall(r'\b\w{4,}\b', text)

    wc = (
        pd.Series(words)
        .value_counts()
        .head(20)
        .reset_index()
    )
    wc.columns = ['palavra', 'freq']

    fig3 = px.treemap(
        wc,
        path=['palavra'],
        values='freq',
        title='Nuvem de Palavras – Títulos'
    )
    fig3.show()

In [12]:
plot_charts(df_db)